In [1]:
import numpy as np

from enterprise_warp import enterprise_warp

from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc
# ─── Monkey-patch EntryPoint.default_kwargs ────────────────────────────────────
try:
    import entrypoints
    entrypoints.EntryPoint.default_kwargs = property(
        lambda self: getattr(self.load(), "__kwdefaults__", {}) or {}
    )
except ImportError:
    pass

try:
    # In case enterprise_warp is using importlib.metadata.EntryPoint instead
    import importlib.metadata as _im
    _im.EntryPoint.default_kwargs = property(
        lambda self: getattr(self.load(), "__kwdefaults__", {}) or {}
    )
except (ImportError, AttributeError):
    pass
# ────────────────────────────────────────────────────────────────────────────────

MPI startup(): FI_PSM3_UUID was not generated, please set it to avoid possible resources ownership conflicts between MPI processes
Optional acor package is not installed. Acor is optionally used to calculate the effective chain length for output in the chain file.


/home/ezahraou/.local/lib/python3.10/site-packages/enterprise_warp-0.0.2-py3.10.egg/enterprise_warp/results.py:32: UserWarning: ChainConsumer is not available


In [2]:
import sys
import ppta_dr2_models


#"gwb": "pol_dist_10_nfreqs"
# keep only the script name
sys.argv = sys.argv[:1]

path_par = "/fred/oz103/ezahraoui/PPTA/run_warp/sampling_params/ppta_pol_selec_7psr_byband_alleccor.dat"
#path_par = "/fred/oz103/ezahraoui/PPTA/run_wrap/sampling_params/ppta_dr2_vanilla_curn_full_param.dat"
#opts = enterprise_warp.parse_commandline()
opts = enterprise_warp.parse_commandline()

custom = ppta_dr2_models.PPTADR2Models



In [3]:
#"pol_dist_5_nfreqs" : 
params = enterprise_warp.Params(path_par,opts=opts,custom_models_obj=custom)


------------------
Setting default parameters with file  /fred/oz103/ezahraoui/PPTA/run_warp/sampling_params/ppta_pol_selec_7psr_byband_alleccor.dat
Setting default Solar System Ephemeris: DE438
Only using pulsars from psrlist
Setting a default linear timing model
Setting timing model SVD to 0 (False)
Including transient events to specific pulsar models
Setting reference radio frequency to 1400 MHz
------------------
Setting sampler kwargs from the parameter file:
------------------
Number of .par files:  25
Number of .tim files:  25
Loading pulsars
[tempo2Util.C:396] Warning: [TIM1] Please place MODE flags in the parameter file 
[tempo2Util.C:401] Warning: [DUP1] duplicated warnings have been suppressed.


Pol calibration is set to True
Adding polynomial distortion vec to pulsars


In [4]:
pta = enterprise_warp.init_pta(params)
print('Pulsar Timing Array: ', len(pta))


GWB/CPL options:  pol_dist_selec_15_nfreqs
Number of Fourier frequencies for the GWB/CPL signal:  15
Adding Pol Selection ORF
returned pol: 
Using noise parameters from the file:  {'J0711-6830_CASPSR_40CM_efac': 1.1298258736280822, 'J0711-6830_CASPSR_40CM_log10_equad': -7.6299166849010085, 'J0711-6830_CPSR2_20CM_efac': 1.064084985404349, 'J0711-6830_CPSR2_20CM_log10_equad': -6.04717097779411, 'J0711-6830_CPSR2_50CM_efac': 1.0940590289422412, 'J0711-6830_CPSR2_50CM_log10_equad': -7.731224036363758, 'J0711-6830_PDFB1_10CM_efac': 0.9877037281298511, 'J0711-6830_PDFB1_10CM_log10_equad': -5.21005052309488, 'J0711-6830_PDFB1_1433_efac': 1.092925238127751, 'J0711-6830_PDFB1_1433_log10_equad': -7.755054181029302, 'J0711-6830_PDFB1_20CM_efac': 1.082837868905749, 'J0711-6830_PDFB1_20CM_log10_equad': -8.07792126238142, 'J0711-6830_PDFB1_early_10CM_efac': 0.9638568848852437, 'J0711-6830_PDFB1_early_10CM_log10_equad': -5.280023683036845, 'J0711-6830_PDFB1_early_20CM_efac': 0.9999217999305334, 'J071

In [5]:
#super_model = hypermodel.HyperModel(pta)

x0 = np.hstack([p.sample() for p in pta[0].params])
ndim = len(x0)
print('ndim: ', ndim)
cov = np.diag(np.ones(ndim) *1**2)
print('Super model parameters: ', pta[0].params)
#sampler = super_model.setup_sampler(resume=True, outdir=params.output_dir)
sampler = ptmcmc(ndim, pta[0].get_lnlikelihood, pta[0].get_lnprior, cov, 
                outDir=params.output_dir, resume=False)
N = int(2e6)

# Remove extra kwargs that Bilby took from PTSampler module, not ".sample"
#ptmcmc_sample_kwargs = inspect.getargspec(sampler.sample).args
# upd_sample_kwargs = {key: val for key, val in params.sampler_kwargs.items()
#                               if key in ptmcmc_sample_kwargs}
#del upd_sample_kwargs['Niter']
#del upd_sample_kwargs['p0']

#sampler.sample(x0, N, **upd_sample_kwargs)
print('len(x0): ', len(x0))


ndim:  103
Super model parameters:  [J0711-6830_10CM_efac:Uniform(pmin=0.0, pmax=10.0), J0711-6830_10CM_log10_ecorr:Uniform(pmin=-10.0, pmax=-5.0), J0711-6830_10CM_log10_tnequad:Uniform(pmin=-10.0, pmax=-5.0), J0711-6830_20CM_efac:Uniform(pmin=0.0, pmax=10.0), J0711-6830_20CM_log10_ecorr:Uniform(pmin=-10.0, pmax=-5.0), J0711-6830_20CM_log10_tnequad:Uniform(pmin=-10.0, pmax=-5.0), J0711-6830_40CM_efac:Uniform(pmin=0.0, pmax=10.0), J0711-6830_40CM_log10_ecorr:Uniform(pmin=-10.0, pmax=-5.0), J0711-6830_40CM_log10_tnequad:Uniform(pmin=-10.0, pmax=-5.0), J0711-6830_dm_gp_gamma:Uniform(pmin=0.0, pmax=10.0), J0711-6830_dm_gp_log10_A:Uniform(pmin=-20.0, pmax=-6.0), J0711-6830_red_noise_gamma:Uniform(pmin=0.0, pmax=10.0), J0711-6830_red_noise_log10_A:Uniform(pmin=-20.0, pmax=-6.0), J1017-7156_10CM_efac:Uniform(pmin=0.0, pmax=10.0), J1017-7156_10CM_log10_ecorr:Uniform(pmin=-10.0, pmax=-5.0), J1017-7156_10CM_log10_tnequad:Uniform(pmin=-10.0, pmax=-5.0), J1017-7156_20CM_efac:Uniform(pmin=0.0, pmax

In [6]:
x0 = np.hstack([p.sample() for p in pta[0].params])
x0

array([  1.97929015,  -5.09985019,  -9.05361764,   9.41208149,
        -8.12136034,  -8.16254654,   9.19190007,  -6.47951488,
        -9.35803461,   7.72995871, -18.81189601,   8.2613266 ,
       -14.30467442,   1.25754371,  -7.51227657,  -5.15651142,
         6.22094365,  -7.77217544,  -9.49047094,   8.94726989,
        -5.18800315,  -6.39351138,   5.23389544,  -6.67869944,
         1.96986336, -11.00635845,   3.39527565,  -6.44806835,
        -5.76483766,   7.01366315,  -5.45908096,  -9.50337713,
         5.00237412,  -9.00656885,  -6.2647911 ,   6.90221446,
       -10.14669707,   2.40777002,  -7.73820158,   9.80071084,
        -8.51949497,  -8.95721342,   6.60224437,  -6.57027324,
        -6.77740796,   1.68134336,  -5.97349344,  -8.53810283,
         1.27845665, -15.8335263 ,   2.70275873, -11.63956326,
         0.16297216,  -7.98788632,  -9.64317016,   6.109591  ,
        -8.28742824,  -8.55095708,   8.22376792,  -5.16730674,
        -8.83277776,   3.51610855, -16.86923026,   6.95

In [14]:
pta[0].get_lnprior(x0)

-215.35545866394816

In [15]:
pta[0].get_lnlikelihood(x0)

-inf

In [ ]:
sampler.sample(x0, N, SCAMweight=30, AMweight=15,thin=20,)

/fred/oz002/ezahraoui/.conda/envs/ppta/lib/python3.10/site-packages/PTMCMCSampler/PTMCMCSampler.py:567: RuntimeWarning: invalid value encountered in scalar subtract
/fred/oz002/ezahraoui/.conda/envs/ppta/lib/python3.10/site-packages/enterprise/signals/parameter.py:70: RuntimeWarning: divide by zero encountered in log


Finished 0.05 percent in 136.978361 s Acceptance rate = 0.422